# `utils_MemberA.ipynb` — Lexical / set-based features

**Owner: Member A.**  This notebook documents the functions in section *2. MEMBER A — Lexical / set-based features* of `utils.py`.

All functions are imported from `utils` and shown on small examples; they are implemented from scratch (no nltk / no sklearn for the feature itself).

In [ ]:
import utils

Q1 = 'How can I learn Python quickly?'
Q2 = 'What is the best way to learn Python fast?'
Q3 = 'How do I cook pasta?'  # unrelated

## Tokenisation
Before any set-based feature, both questions are lower-cased and stripped of punctuation by `utils.normalise` and tokenised on white space.

In [ ]:
print(utils.normalise(Q1))
print(utils.tokenize(Q1))
print(utils.content_tokens(Q1))   # stop-words removed

## `jaccard_similarity(text1, text2)`
$$\mathrm{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
Treats each question as a *set* of unique tokens.  Symmetric, in $[0, 1]$.  A value of 1 means identical vocabularies (regardless of word order or repetition); 0 means no shared word.

In [ ]:
print('paraphrase :', utils.jaccard_similarity(Q1, Q2))
print('unrelated  :', utils.jaccard_similarity(Q1, Q3))

## `jaccard_content(text1, text2)`
Same Jaccard formula but **after removing stop words**.  Function words like *the*, *is*, *what* dominate raw Jaccard scores between two questions, so this version focuses on content words.

In [ ]:
print('paraphrase :', utils.jaccard_content(Q1, Q2))
print('unrelated  :', utils.jaccard_content(Q1, Q3))

## `word_overlap_ratio(text1, text2)`
$$\mathrm{overlap}(A, B) = \frac{|A \cap B|}{\min(|A|, |B|)}$$
Different from Jaccard: instead of dividing by the *union* it divides by the *shorter* set.  This makes it robust to length asymmetry — useful when one duplicate is a longer rewrite of the other.

In [ ]:
print(utils.word_overlap_ratio(Q1, Q2))
print(utils.word_overlap_ratio('How to cook?', 'How to cook pasta carbonara at home?'))

## `length_features` and `char_length_features`
Each returns `[len(q1), len(q2), |len(q1)-len(q2)|, min/max]` — first in tokens, then in characters.  Empirically two duplicates tend to have similar lengths; very different lengths are weak evidence of *non*-duplication.

In [ ]:
print('token  features:', utils.length_features(Q1, Q2))
print('char   features:', utils.char_length_features(Q1, Q2))

## `common_words_count(text1, text2)`
Raw count `|A ∩ B|` of unique tokens shared between the two questions — kept alongside the ratios because gradient-boosted trees benefit from having both the absolute count and the normalised version available.

In [ ]:
print(utils.common_words_count(Q1, Q2))

## Summary
These features provide a fast first-order similarity signal.  Their main weakness is that they cannot detect **semantic** equivalence (`automobile` vs `car`).  That gap is filled in part by the TF-IDF cosine of Member C and by Member B's edit-distance features when the paraphrase reuses *similar* words rather than identical ones.